In [2]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
files = [
    r"C:\Users\linji\OneDrive\Desktop\IDXExchange\ca-price-prediction\data\CRMLSSold202512.csv",
    r"C:\Users\linji\OneDrive\Desktop\IDXExchange\ca-price-prediction\data\CRMLSSold202601.csv",
    r"C:\Users\linji\OneDrive\Desktop\IDXExchange\ca-price-prediction\data\CRMLSSold202602.csv",
    r"C:\Users\linji\OneDrive\Desktop\IDXExchange\ca-price-prediction\data\CRMLSSold202603.csv",
    r"C:\Users\linji\OneDrive\Desktop\IDXExchange\ca-price-prediction\data\CRMLSSold202604.csv",
    r"C:\Users\linji\OneDrive\Desktop\IDXExchange\ca-price-prediction\data\CRMLSSold202605.csv",
]

data = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


C:\Users\linji\AppData\Local\Temp\ipykernel_30692\3509414693.py:10: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


In [4]:

# Filter to Single Family Residences only
data = data[
    (data["PropertyType"] == "Residential") &
    (data["PropertySubType"] == "SingleFamilyResidence")
].copy()

print(f"Rows: {len(data)}")
print(f"Columns: {len(data.columns)}")

Rows: 61727
Columns: 78


In [5]:
# Calculate missing percentage for every column
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(1)

missing_summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing %", ascending=False)

# Show all columns with ANY missing values
print(missing_summary[missing_summary["Missing Count"] > 0].to_string())

                              Missing Count  Missing %
BusinessType                          61727      100.0
CoveredSpaces                         61727      100.0
MiddleOrJuniorSchoolDistrict          61727      100.0
FireplacesTotal                       61727      100.0
TaxAnnualAmount                       61727      100.0
AboveGradeFinishedArea                61727      100.0
TaxYear                               61727      100.0
ElementarySchoolDistrict              61727      100.0
WaterfrontYN                          61691       99.9
BelowGradeFinishedArea                61282       99.3
BasementYN                            60272       97.6
BuilderName                           58946       95.5
LotSizeDimensions                     57916       93.8
BuildingAreaTotal                     57625       93.4
CoBuyerAgentFirstName                 55918       90.6
ElementarySchool                      53864       87.3
MiddleOrJuniorSchool                  53872       87.3
HighSchool

In [6]:
# drop columns that are 100% empty — completely useless
fully_empty = [
    "BusinessType", "CoveredSpaces", "MiddleOrJuniorSchoolDistrict",
    "FireplacesTotal", "TaxAnnualAmount", "AboveGradeFinishedArea",
    "TaxYear", "ElementarySchoolDistrict", "BelowGradeFinishedArea"
]

# drop columns over 60% missing — too sparse to be reliable
over_60_missing = [
    "WaterfrontYN", "BasementYN", "BuilderName", "LotSizeDimensions",
    "BuildingAreaTotal", "CoBuyerAgentFirstName", "ElementarySchool",
    "MiddleOrJuniorSchool", "HighSchool", "CoListAgentFirstName",
    "CoListAgentLastName", "CoListOfficeName", "AssociationFeeFrequency",
    "SubdivisionName"
]

# drop admin/identity columns — agent names, emails, IDs, not useful for modeling
admin_cols = [
    "ListAgentFirstName", "ListAgentLastName", "ListAgentFullName",
    "ListAgentEmail", "ListAgentAOR",
    "BuyerAgentFirstName", "BuyerAgentLastName", "BuyerAgentMlsId",
    "BuyerAgentAOR", "BuyerOfficeName", "BuyerOfficeAOR",
    "UnparsedAddress", "StreetNumberNumeric", "PurchaseContractDate"
]

# Combine all and drop
drop_cols = list(set(fully_empty + over_60_missing + admin_cols))
data.drop(columns=[col for col in drop_cols if col in data.columns], inplace=True)

# also drop StateOrProvince — all rows are CA, zero information
if "StateOrProvince" in data.columns:
    data.drop(columns=["StateOrProvince"], inplace=True)

print(f"Columns remaining: {len(data.columns)}")
print(list(data.columns))

Columns remaining: 40
['Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'CloseDate', 'ClosePrice', 'Latitude', 'Longitude', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'ListingKeyNumeric', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'YearBuilt', 'ListingId', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'ContractStatusChangeDate', 'ListingContractDate', 'FireplaceYN', 'Stories', 'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee', 'LotSizeSquareFeet']


In [7]:
# Identifier columns — just row IDs, no predictive value
id_cols = ["ListingKey", "ListingKeyNumeric", "ListingId"]

# Already filtered — all rows are same value, zero information
already_filtered = ["PropertyType", "PropertySubType", "MlsStatus"]

# Redundant location — we already have LotSizeSquareFeet which is cleaner
redundant = ["LotSizeAcres", "LotSizeArea", "MainLevelBedrooms"]

# Date columns — we only need CloseDate for the train/test split
date_cols = ["ContractStatusChangeDate", "ListingContractDate"]

# Text columns with too many unique values — hard to encode meaningfully
high_cardinality = ["ListOfficeName", "Flooring", "Levels", "MLSAreaMajor"]

drop_cols2 = list(set(id_cols + already_filtered + redundant + date_cols + high_cardinality))
data.drop(columns=[col for col in drop_cols2 if col in data.columns], inplace=True)

print(f"Columns remaining: {len(data.columns)}")
print(list(data.columns))

Columns remaining: 25
['ViewYN', 'PoolPrivateYN', 'OriginalListPrice', 'CloseDate', 'ClosePrice', 'Latitude', 'Longitude', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'CountyOrParish', 'AttachedGarageYN', 'ParkingTotal', 'YearBuilt', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN', 'Stories', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee', 'LotSizeSquareFeet']


In [8]:
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(1)

missing_summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing %", ascending=False)

print(missing_summary[missing_summary["Missing Count"] > 0].to_string())

                       Missing Count  Missing %
AssociationFee                 17616       28.5
HighSchoolDistrict             16629       26.9
AttachedGarageYN                7527       12.2
Stories                         6493       10.5
ViewYN                          5295        8.6
PoolPrivateYN                   4833        7.8
NewConstructionYN               4686        7.6
GarageSpaces                    2353        3.8
LotSizeSquareFeet               1081        1.8
OriginalListPrice                136        0.2
YearBuilt                         36        0.1
FireplaceYN                       58        0.1
LivingArea                        30        0.0
Longitude                          9        0.0
Latitude                           9        0.0
City                              14        0.0
BathroomsTotalInteger              1        0.0
PostalCode                         1        0.0


In [9]:
# Boolean YN columns → False
yn_cols = ["AttachedGarageYN", "ViewYN", "PoolPrivateYN", 
           "NewConstructionYN", "FireplaceYN"]
for col in yn_cols:
    data[col] = data[col].fillna(False)

# AssociationFee → 0
data["AssociationFee"] = data["AssociationFee"].fillna(0)

# GarageSpaces → 0
data["GarageSpaces"] = data["GarageSpaces"].fillna(0)

# OriginalListPrice → ListPrice
data["OriginalListPrice"] = data["OriginalListPrice"].fillna(data["ListPrice"])

# Simple median fills
for col in ["Stories", "YearBuilt", "LivingArea", "BathroomsTotalInteger", "LotSizeSquareFeet"]:
    data[col] = data[col].fillna(data[col].median())

# County-specific median for Latitude, Longitude, LotSizeSquareFeet
for col in ["Latitude", "Longitude"]:
    county_medians = data.groupby("CountyOrParish")[col].median()
    data[col] = data[col].fillna(data["CountyOrParish"].map(county_medians))
    data[col] = data[col].fillna(data[col].median())  # fallback to global median

# Categorical → Unknown
data["HighSchoolDistrict"] = data["HighSchoolDistrict"].fillna("Unknown")
data["City"] = data["City"].fillna("Unknown")
data["PostalCode"] = data["PostalCode"].fillna("Unknown")

# Confirm no missing values remain
remaining_missing = data.isnull().sum().sum()
print(f"Total missing values remaining: {remaining_missing}")
print(f"Rows: {len(data)}, Columns: {len(data.columns)}")

Total missing values remaining: 0
Rows: 61727, Columns: 25


In [10]:
# Convert CloseDate to datetime
data["CloseDate"] = pd.to_datetime(data["CloseDate"], errors="coerce")

# Extract month period for splitting
data["CloseMonth"] = data["CloseDate"].dt.to_period("M")

# Check what months we have
print("Months in dataset:")
print(data["CloseMonth"].value_counts().sort_index())

Months in dataset:
CloseMonth
2025-12    10455
2026-01     7490
2026-02     8550
2026-03    11177
2026-04    12031
2026-05    12024
Freq: M, Name: count, dtype: int64


In [11]:
from sklearn.preprocessing import LabelEncoder

# Columns that need encoding
cat_cols = ["City", "PostalCode", "CountyOrParish", "HighSchoolDistrict"]

le = LabelEncoder()
for col in cat_cols:
    data[col + "_encoded"] = le.fit_transform(data[col].astype(str))

# Convert boolean YN columns to 0/1
yn_cols = ["AttachedGarageYN", "ViewYN", "PoolPrivateYN", 
           "NewConstructionYN", "FireplaceYN"]
for col in yn_cols:
    data[col] = data[col].astype(int)

print("Encoded columns added:")
print([c for c in data.columns if "_encoded" in c])
print("\nSample of encoded values:")
print(data[["City", "City_encoded", "PostalCode", "PostalCode_encoded"]].drop_duplicates().head(10))

Encoded columns added:
['City_encoded', 'PostalCode_encoded', 'CountyOrParish_encoded', 'HighSchoolDistrict_encoded']

Sample of encoded values:
              City  City_encoded PostalCode  PostalCode_encoded
0     Walnut Creek           863      94596                1129
2   Woodland Hills           902      91364                 220
3         San Jose           715      95121                1291
7         San Jose           715      95124                1295
9         San Jose           715      95128                1299
10   Mission Viejo           497      92692                 568
11         Concord           172      94519                 951
13        Temecula           812      92592                 526
14   San Francisco           712      94112                 867
16    Laguna Beach           391      92651                 550


In [12]:
most_recent_month = data["CloseMonth"].max()

# Training window — 5 months preceding the test month
n_months_train = 5
training_months = pd.period_range(
    end=most_recent_month - 1, 
    periods=n_months_train, 
    freq="M"
)

train_data = data[data["CloseMonth"].isin(training_months)].copy()
test_data  = data[data["CloseMonth"] == most_recent_month].copy()

print(f"Test month: {most_recent_month}")
print(f"Training months: {list(training_months)}")
print(f"Train rows: {len(train_data)}")
print(f"Test rows:  {len(test_data)}")

Test month: 2026-05
Training months: [Period('2025-12', 'M'), Period('2026-01', 'M'), Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M')]
Train rows: 49703
Test rows:  12024


In [13]:
# Final feature columns for modeling
# REMOVE these two lines from feature_cols
#"ListPrice"
#"OriginalListPrice"
feature_cols = [
    # Size & structure
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "Stories",
    # Parking
    "GarageSpaces", "ParkingTotal",
    # Amenities
    "AttachedGarageYN", "FireplaceYN", "PoolPrivateYN",
    "ViewYN", "NewConstructionYN",
    # Market context
    "DaysOnMarket", "AssociationFee",
    # Location
    "Latitude", "Longitude",
    "City_encoded", "PostalCode_encoded",
    "CountyOrParish_encoded", "HighSchoolDistrict_encoded",
]
# Confirm no missing values in features
print("Missing values in features:")
print(train_data[feature_cols].isnull().sum().sum())

# Check shapes
X_train = train_data[feature_cols]
y_train = train_data["ClosePrice"]

X_test = test_data[feature_cols]
y_test = test_data["ClosePrice"]

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train range: ${y_train.min():,.0f} — ${y_train.max():,.0f}")
print(f"y_test range:  ${y_test.min():,.0f} — ${y_test.max():,.0f}")

Missing values in features:
0

X_train shape: (49703, 21)
X_test shape:  (12024, 21)
y_train range: $2 — $796,000,000
y_test range:  $11,900 — $97,972,500


In [14]:
output_path = r"C:\Users\linji\OneDrive\Desktop\IDXExchange\ca-price-prediction\data\cleaned_data_v2.csv"
data.to_csv(output_path, index=False)

train_path = r"C:\Users\linji\OneDrive\Desktop\IDXExchange\ca-price-prediction\data\train_data.csv"
test_path  = r"C:\Users\linji\OneDrive\Desktop\IDXExchange\ca-price-prediction\data\test_data.csv"

train_data.to_csv(train_path, index=False)
test_data.to_csv(test_path, index=False)

print("Files saved successfully!")
print(f"Full cleaned data:  {data.shape[0]} rows, {data.shape[1]} columns")
print(f"Train set:          {train_data.shape[0]} rows")
print(f"Test set:           {test_data.shape[0]} rows")
print(f"\nFeatures ready for modeling: {len(feature_cols)}")
print(feature_cols)

Files saved successfully!
Full cleaned data:  61727 rows, 30 columns
Train set:          49703 rows
Test set:           12024 rows

Features ready for modeling: 21
['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'Stories', 'GarageSpaces', 'ParkingTotal', 'AttachedGarageYN', 'FireplaceYN', 'PoolPrivateYN', 'ViewYN', 'NewConstructionYN', 'DaysOnMarket', 'AssociationFee', 'Latitude', 'Longitude', 'City_encoded', 'PostalCode_encoded', 'CountyOrParish_encoded', 'HighSchoolDistrict_encoded']
